# Scalars and empty tensors

A tensor with no axes can still contain a value. A tensor with a zero-length axis contains none. Work through the diagrams below, then distinguish an empty group from an empty output.

Install `rainbow-tensor` in this notebook kernel's environment to run the examples and focus controls.


In [ ]:
import numpy as np
from IPython.display import display

import rainbow_tensor as rt

scalar = np.array(7)
vector = np.array([7])
empty_vector = np.empty((0,))

for array in (scalar, vector, empty_vector):
    print(f'Shape {array.shape}, elements {array.size}')
    display(rt.shape(array))

assert (scalar.shape, scalar.size) == ((), 1)
assert (vector.shape, vector.size) == ((1,), 1)
assert (empty_vector.shape, empty_vector.size) == ((0,), 0)


## One scalar, one coordinate

The scalar has one real cell at coordinate `()`, with flat index 0. It has no axis to label. Indexing with `()` keeps that value, and the result panel retains logical shape `()`.


In [ ]:
indexed = rt.index(scalar, (), show_result=True)
display(indexed)
assert indexed.result_shape == ()
assert indexed.index_mapping.source_coord(()) == ()
assert list(indexed.selected) == [()]


## A zero anywhere makes the source empty

Shape `(2, 0, 3)` contains zero elements. The view shows its shape and a `No elements` marker without fabricating a numeric cell. A bare shape tuple supplies generated values rather than an allocated array, so `rt.shape(())` displays a scalar placeholder of 0.


In [ ]:
empty = np.empty((2, 0, 3))
display(rt.shape(empty))
display(rt.shape((1_000_000, 0, 1_000_000)))
display(rt.shape(()))
assert empty.size == 0


## Six outputs, each with no contributions

Predict the result of reducing axis 1 of `(2, 0, 3)`. Removing that axis leaves `(2, 3)`, so there are six output cells. Each combines zero source values. The empty sum is 0. The empty mean is NaN because an average of no observations is undefined.


In [ ]:
empty_sums = rt.sum(empty, axis=1, focus=(1, 2))
empty_means = rt.mean(empty, axis=1, focus=(1, 2))
display(empty_sums)
display(empty_means)

assert empty_sums.result_shape == empty_means.result_shape == (2, 3)
assert empty_sums.trace.output_coord == (1, 2)
assert empty_sums.trace.term_count == empty_means.trace.term_count == 0
assert empty_sums.trace.terms == empty_means.trace.terms == ()
np.testing.assert_array_equal(empty.sum(axis=1), np.zeros((2, 3)))


## No output cells

Now reduce axis 2 instead. Shape `(2, 0, 3)` becomes `(2, 0)`, which is still empty. There is no output value to focus, so the visual has `trace=None`. Leave `focus=None`. An explicit coordinate is invalid.

Keeping an axis at length one preserves the same distinction. Reducing the zero-length axis with `keepdims=True` gives `(2, 1, 3)` and six zeros.


In [ ]:
empty_result = rt.sum(empty, axis=2)
display(empty_result)
assert empty_result.result_shape == (2, 0)
assert empty_result.trace is None

kept = rt.sum(empty, axis=1, keepdims=True)
display(kept)
assert kept.result_shape == (2, 1, 3)

try:
    rt.sum(empty, axis=2, focus=(0, 0))
except IndexError as error:
    print(f'No output coordinate exists: {error}')
else:
    raise AssertionError('An empty result must reject explicit focus')


## Reducing a scalar and reducing no axes

Omitting `axis` reduces all axes. A scalar already has no axes, so its value and shape `()` remain. Passing `axis=()` asks to reduce no axes for any input. Applied to an empty source, it preserves the empty shape.


In [ ]:
scalar_sum = rt.sum(scalar)
scalar_mean = rt.mean(scalar, focus=())
display(scalar_sum)
display(scalar_mean)
assert scalar_sum.result_shape == scalar_mean.result_shape == ()
assert scalar_sum.trace.output_coord == ()
assert scalar_sum.trace.term_count == 1

identity = rt.sum(empty, axis=())
display(identity)
assert identity.result_shape == empty.shape
assert identity.trace is None


## Reshape still preserves the element count

A scalar can become a one-element vector. An empty source can take another zero-element shape. Inferring `-1` works when the known dimensions have a nonzero product. In `(0, -1)`, every inferred size would produce zero elements, so the target is ambiguous and rejected.


In [ ]:
display(rt.reshape(scalar, (1,)))
display(rt.reshape(vector, ()))

inferred = rt.reshape(empty, (-1, 3))
explicit = rt.reshape(empty, (2, 0))
display(inferred)
display(explicit)
assert inferred.result_shape == (0, 3)
assert explicit.result_shape == (2, 0)

try:
    rt.reshape(empty, (0, -1))
except ValueError as error:
    print(f'Ambiguous target: {error}')
else:
    raise AssertionError('A zero dimension with -1 must be rejected')


## Matrix products can have an empty inner axis

The product `(2, 0) @ (0, 3)` has shape `(2, 3)`. Every output is a sum of zero products, so every value is 0. Matmul still requires operands with at least one axis. A scalar is not a valid matmul operand.


In [ ]:
left = np.empty((2, 0))
right = np.empty((0, 3))
product = rt.matmul(left, right, focus=(1, 2))
display(product)
assert product.result_shape == (2, 3)
assert product.trace.term_count == 0
np.testing.assert_array_equal(left @ right, np.zeros((2, 3)))

try:
    rt.matmul(scalar, vector)
except ValueError as error:
    print(f'Scalar matmul is invalid: {error}')
else:
    raise AssertionError('Matmul must reject a scalar operand')


## Focus controls

An empty output has no coordinate fields and a disabled **Update focus** button. Its static figure remains available. A scalar output also has no coordinate fields, but it has a real coordinate `()` and an enabled update button.

Live controls need a running notebook kernel and widget support in its host. For hosts without widget support, use the static figures above.


In [ ]:
empty_explorer = rt.explore(rt.sum, empty, axis=2)
scalar_explorer = rt.explore(rt.sum, scalar)
display(empty_explorer)
display(scalar_explorer)
assert empty_explorer.focus is None
assert empty_explorer.visual.trace is None
assert empty_explorer.coordinates == ()
assert empty_explorer.update_button.disabled
assert scalar_explorer.focus == ()
assert not scalar_explorer.update_button.disabled


Run this cell when you finish with the live controls. Static SVG figures from earlier cells remain available.


In [ ]:
for explorer in (empty_explorer, scalar_explorer):
    explorer.close()


## Read the three different displays

- **No elements** means that the tensor has no cells.
- **NaN** in an empty mean occupies a real output cell whose average is undefined.
- **?** means that an output value was skipped under the calculation budgets.

The existing `max_terms` and `max_total_terms` limits still apply. An empty contribution group costs zero terms. An empty output has zero planned output cells. Numerical previews continue to use Python scalar arithmetic, so use the original array backend when its dtype and accumulation behavior matter.
